# 042926 TIFF batch preprocessing

180ページのTIFFを一度にメモリへ読み込まず、1ページずつ前処理して保存します。背景補正は基準値100・最大ゲイン3で行い、結果を100倍（実効スケール10000）してから `0–65535` にクリップし、`uint16` に変換します。正規化は行いません。

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import tifffile as tiff

# VS Code/Jupyterの起動位置がリポジトリ直下でもnotebook直下でも動くようにする
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Project root containing src/ was not found.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from preprocess import preprocess_image

INPUT_TIF = PROJECT_ROOT / 'data' / '042926_MAY08R_FOS_1_retake_c.tif'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / (INPUT_TIF.stem + '_uint16_scale10000')
OVERWRITE = False  # 中断後の再実行では、保存済みページをスキップする
OUTPUT_SCALE = 100.0  # 補正基準値100 × 出力100倍 = 実効スケール10000

PARAMS = {
    'shading_sigma': 120,
    'shading_reference_level': 100,
    'shading_max_gain': 3.0,
    'stripe_sigma': (128, 256),
    'stripe_level': 7,
    'stripe_wavelet': 'db2',
    'bg_radius': 20,
}

print('Input :', INPUT_TIF)
print('Output:', OUTPUT_DIR)

Input : C:\workspace\LSFM_pp\data\042926_MAY08R_FOS_1_retake_c.tif
Output: C:\workspace\LSFM_pp\outputs\042926_MAY08R_FOS_1_retake_c


In [2]:
# 全体を読み込まず、メタデータのみ確認
with tiff.TiffFile(str(INPUT_TIF)) as tif:
    page_count = len(tif.pages)
    first_page = tif.pages[0]
    print('Pages:', page_count)
    print('Page shape:', first_page.shape)
    print('Input dtype:', first_page.dtype)

assert page_count == 180, 'Expected 180 pages, found {}'.format(page_count)

Pages: 180
Page shape: (4096, 2160)
Input dtype: uint16


In [3]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
started_at = time.time()
saved = 0
skipped = 0

with tiff.TiffFile(str(INPUT_TIF)) as tif:
    total = len(tif.pages)
    for page_index, page in enumerate(tif.pages):
        page_number = page_index + 1
        output_path = OUTPUT_DIR / '{}_{:03d}.tif'.format(INPUT_TIF.stem, page_number)

        if output_path.exists() and not OVERWRITE:
            skipped += 1
            print('[{}/{}] skipped: {}'.format(page_number, total, output_path.name))
            continue

        raw = page.asarray()
        result = preprocess_image(raw, return_intermediates=False, **PARAMS)

        # 正規化せず100倍し、uint16の範囲外だけを切る
        scaled = result['preprocessed'] * OUTPUT_SCALE
        output_uint16 = np.clip(scaled, 0, 65535).astype(np.uint16)
        tiff.imwrite(str(output_path), output_uint16)
        saved += 1
        print('[{}/{}] saved : {}'.format(page_number, total, output_path.name))

elapsed = time.time() - started_at
print('Done: saved={}, skipped={}, elapsed={:.1f} min'.format(saved, skipped, elapsed / 60.0))

[1/180] saved : 042926_MAY08R_FOS_1_retake_c_001.tif
[2/180] saved : 042926_MAY08R_FOS_1_retake_c_002.tif
[3/180] saved : 042926_MAY08R_FOS_1_retake_c_003.tif
[4/180] saved : 042926_MAY08R_FOS_1_retake_c_004.tif
[5/180] saved : 042926_MAY08R_FOS_1_retake_c_005.tif
[6/180] saved : 042926_MAY08R_FOS_1_retake_c_006.tif
[7/180] saved : 042926_MAY08R_FOS_1_retake_c_007.tif
[8/180] saved : 042926_MAY08R_FOS_1_retake_c_008.tif
[9/180] saved : 042926_MAY08R_FOS_1_retake_c_009.tif
[10/180] saved : 042926_MAY08R_FOS_1_retake_c_010.tif
[11/180] saved : 042926_MAY08R_FOS_1_retake_c_011.tif
[12/180] saved : 042926_MAY08R_FOS_1_retake_c_012.tif
[13/180] saved : 042926_MAY08R_FOS_1_retake_c_013.tif
[14/180] saved : 042926_MAY08R_FOS_1_retake_c_014.tif
[15/180] saved : 042926_MAY08R_FOS_1_retake_c_015.tif
[16/180] saved : 042926_MAY08R_FOS_1_retake_c_016.tif
[17/180] saved : 042926_MAY08R_FOS_1_retake_c_017.tif
[18/180] saved : 042926_MAY08R_FOS_1_retake_c_018.tif
[19/180] saved : 042926_MAY08R_FOS_1_

KeyboardInterrupt: 

In [ ]:
# 保存結果の枚数・shape・dtype・値域を確認
output_files = sorted(OUTPUT_DIR.glob('*.tif'))
print('Output files:', len(output_files))

if output_files:
    sample = tiff.imread(str(output_files[0]))
    print('First output:', output_files[0].name)
    print('Shape:', sample.shape)
    print('Dtype:', sample.dtype)
    print('Value range:', int(sample.min()), '-', int(sample.max()))

assert len(output_files) == 180, 'Expected 180 output files, found {}'.format(len(output_files))
assert sample.dtype == np.uint16